<a href="https://colab.research.google.com/github/Piyush-08-bot/pytorch/blob/main/Training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install PyTorch if needed
%pip install torch torchvision torchaudio --quiet
print('✓ PyTorch installed successfully!')

# PyTorch Training Pipeline

A comprehensive guide to building and training neural networks with PyTorch.

## 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from tqdm import tqdm

# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 2. Generate Sample Data

In [ ]:
# Create synthetic data for binary classification
np.random.seed(42)
torch.manual_seed(42)

# Generate data
n_samples = 1000
n_features = 20

X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, 2, n_samples)

# Convert to tensors
X_tensor = torch.FloatTensor(X)
y_tensor = torch.LongTensor(y)

# Split into train and validation
train_size = int(0.8 * len(X_tensor))
val_size = len(X_tensor) - train_size

X_train, X_val = X_tensor[:train_size], X_tensor[train_size:]
y_train, y_val = y_tensor[:train_size], y_tensor[train_size:]

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Features: {n_features}")

## 3. Create DataLoaders

In [ ]:
# Create datasets
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 4. Define Model Architecture

In [ ]:
class BinaryClassifier(nn.Module):
    """Neural network for binary classification."""
    
    def __init__(self, input_size=20, hidden_size=64):
        super(BinaryClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size, 32)
        self.fc3 = nn.Linear(32, 2)  # Binary classification (2 classes)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Create model
model = BinaryClassifier(input_size=n_features, hidden_size=64)
model = model.to(device)

print(f"Model:\n{model}")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Setup Loss and Optimizer

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("✓ Loss function: CrossEntropyLoss")
print("✓ Optimizer: Adam (lr=0.001)")
print("✓ Scheduler: StepLR")

## 6. Training and Validation Functions

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # Forward pass
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(batch_y).sum().item()
        total += batch_y.size(0)
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def validate(model, val_loader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(batch_y).sum().item()
            total += batch_y.size(0)
    
    avg_loss = total_loss / len(val_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

print("✓ Training and validation functions defined")

## 7. Train the Model

In [ ]:
# Training loop
num_epochs = 10
best_val_accuracy = 0
patience = 5
patience_counter = 0

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Update scheduler
    scheduler.step()
    
    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Early stopping
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
    
    print(f"Epoch {epoch+1:2d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

print("\n✓ Training completed!")

## 8. Load Best Model and Evaluate

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pth'))

# Final evaluation
final_train_loss, final_train_acc = validate(model, train_loader, criterion, device)
final_val_loss, final_val_acc = validate(model, val_loader, criterion, device)

print("\n=== Final Results ===")
print(f"Train Loss: {final_train_loss:.4f} | Train Accuracy: {final_train_acc:.2f}%")
print(f"Val Loss: {final_val_loss:.4f} | Val Accuracy: {final_val_acc:.2f}%")

## 9. Save Model for Inference

In [ ]:
# Save model architecture and weights
model_path = 'trained_model.pth'
torch.save(model.state_dict(), model_path)
print(f"✓ Model saved to {model_path}")

# Save complete model
torch.save(model, 'complete_model.pth')
print(f"✓ Complete model saved to complete_model.pth")

## 10. Make Predictions

In [ ]:
# Make predictions on new data
def predict(model, X, device):
    model.eval()
    with torch.no_grad():
        if isinstance(X, np.ndarray):
            X = torch.FloatTensor(X)
        X = X.to(device)
        outputs = model(X)
        probabilities = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)
    return predictions.cpu().numpy(), probabilities.cpu().numpy()

# Test on sample
sample_data = X_val[:5].cpu().numpy()
predictions, probs = predict(model, sample_data, device)

print("Sample Predictions:")
for i, (pred, prob) in enumerate(zip(predictions, probs)):
    print(f"Sample {i+1}: Prediction={pred}, Confidence={prob[pred]:.2%}")

## 11. Plot Training History

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o')
axes[1].plot(history['val_acc'], label='Val Accuracy', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
plt.savefig('training_history.png', dpi=100)
print("✓ Training history plot saved")

## 12. Key Training Tips

- **Batch Normalization**: Add `nn.BatchNorm1d()` for faster convergence
- **Dropout**: Helps prevent overfitting
- **Learning Rate Scheduling**: Gradually reduce LR for better convergence
- **Early Stopping**: Stop training when validation accuracy plateaus
- **Data Augmentation**: Essential for image data
- **Checkpointing**: Save best model during training
- **Mixed Precision**: Use `torch.cuda.amp` for faster training on GPUs

<a href="https://colab.research.google.com/github/Piyush-08-bot/pytorch/blob/main/pytorch_training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>